## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import PROCESSED_PATH
from src.selection import (
    MIN_RETURN_OBSERVATIONS,
    percentile_score,
    role_score,
    compute_quant_score,
    compute_selection_score,
    PROTECTED_ROLE_SCORE_CUTOFF,
    REVIEW_SCORE_CUTOFF,
    CANDIDATE_SCORE_CUTOFF,
    MATERIAL_WEIGHT,
    assign_decision,
)

## 2. Paths & Load Data

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

asset_universe_metrics = pd.read_csv(
    DATA_PATH / "asset_universe_metrics.csv"
)

asset_universe_overview = pd.read_csv(
    DATA_PATH / "asset_universe_overview.csv"
)

portfolio_dashboard = pd.read_csv(
    DATA_PATH / "portfolio_dashboard.csv"
)

asset_universe_metrics.head()

In [ ]:
asset_universe_metrics.columns.tolist()

## 3. Prepare Selection Dataset

In [ ]:
selection = asset_universe_metrics.copy()

selection = selection[
    [
        "yf_ticker",
        "name",
        "asset_class",
        "region",
        "role",
        "role_score",
        "source",
        "data_start",
        "data_end",
        "n_return_observations",
        "annualized_return",
        "annualized_volatility",
        "max_drawdown",
        "sharpe_ratio"
    ]
].copy()

selection.head()

## 4. Add Current Portfolio Information

In [ ]:
current_portfolio_info = portfolio_dashboard[
    [
        "yf_ticker",
        "market_value",
        "actual_weight"
    ]
].copy()

selection = selection.merge(
    current_portfolio_info,
    on="yf_ticker",
    how="left"
)

selection["is_current_holding"] = selection["actual_weight"].notna()

selection["market_value"] = selection["market_value"].fillna(0)
selection["actual_weight"] = selection["actual_weight"].fillna(0)

selection[
    [
        "name",
        "source",
        "is_current_holding",
        "market_value",
        "actual_weight"
    ]
]

## 5. Data Quality Filter

In [ ]:
selection["data_quality_ok"] = (
    selection["n_return_observations"] >= MIN_RETURN_OBSERVATIONS
)

selection[
    [
        "name",
        "source",
        "n_return_observations",
        "data_quality_ok"
    ]
].sort_values("n_return_observations")

## 6. Scoring Function

The scoring helpers live in `src/selection.py` (imported in the setup cell, covered by `tests/test_selection.py`):

- `percentile_score` — ranks each metric across the universe (0–1), so metrics on different scales become comparable.
- `compute_quant_score` — **0.50 × Sharpe + 0.30 × Drawdown + 0.20 × Volatility**. Raw return is not a separate component, since Sharpe already contains it. All metrics come from notebook 04 over one common window (short histories back-filled with a same-index proxy).
- `compute_selection_score` — **0.60 × Quant Score + 0.40 × Role Score**, where the role score (0–1) is the asset's strategic importance, set per asset in `data/manual/asset_universe_manual.csv`.
- `assign_decision` — turns the score into a decision bucket via the cutoffs (protected role ≥ 0.80, review ≤ 0.40 for material holdings, candidate ≥ 0.60 for watchlist assets).

## 7. Quantitative Scoring

In [ ]:
# No separate return score: Sharpe already contains return (see
# src/selection.py for the weighting rationale).
selection["sharpe_score"] = percentile_score(
    selection["sharpe_ratio"],
    higher_is_better=True
)

selection["volatility_score"] = percentile_score(
    selection["annualized_volatility"],
    higher_is_better=False
)

selection["drawdown_score"] = percentile_score(
    selection["max_drawdown"],
    higher_is_better=True
)

## 8. Strategic Role Score

In [ ]:
# role_score comes per asset from data/manual/asset_universe_manual.csv.
# Missing values fall back to the generic role map in src/selection.py —
# flagged loudly so every new asset gets a deliberate score.
missing_role_score = selection["role_score"].isna()

if missing_role_score.any():
    print(
        "WARNING: no role_score in asset_universe_manual.csv for: "
        + ", ".join(selection.loc[missing_role_score, "name"])
        + " -> using generic fallback. Set it explicitly in the CSV."
    )

selection.loc[missing_role_score, "role_score"] = (
    selection.loc[missing_role_score, "role"].apply(role_score)
)

selection[
    [
        "name",
        "role",
        "role_score"
    ]
]

## 9. Final Selection Score

In [ ]:
selection["quant_score"] = compute_quant_score(
    selection["sharpe_score"],
    selection["drawdown_score"],
    selection["volatility_score"]
)

selection["selection_score"] = compute_selection_score(
    selection["quant_score"],
    selection["role_score"]
)

In [ ]:
selection.loc[
    ~selection["data_quality_ok"],
    "selection_score"
] = np.nan

In [ ]:
selection_ranked = (
    selection
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

selection_ranked[
    [
        "name",
        "source",
        "role",
        "quant_score",
        "selection_score",
        "data_quality_ok"
    ]
]

## 10. Decision Buckets

In [ ]:
selection_ranked["is_protected_role"] = (
    selection_ranked["role_score"] >= PROTECTED_ROLE_SCORE_CUTOFF
)

selection_ranked["decision"] = selection_ranked.apply(
    assign_decision,
    axis=1
)

selection_ranked[
    [
        "name",
        "role",
        "role_score",
        "is_protected_role",
        "is_current_holding",
        "actual_weight",
        "quant_score",
        "selection_score",
        "decision"
    ]
]

## 11. Display Table

In [ ]:
selection_display = selection_ranked[
    [
        "name",
        "asset_class",
        "region",
        "role",
        "source",
        "is_current_holding",
        "is_protected_role",
        "actual_weight",
        "annualized_return",
        "annualized_volatility",
        "max_drawdown",
        "sharpe_ratio",
        "quant_score",
        "role_score",
        "selection_score",
        "decision"
    ]
].copy()

selection_display

In [ ]:
selection_display_formatted = selection_display.copy()

percentage_cols = [
    "actual_weight",
    "annualized_return",
    "annualized_volatility",
    "max_drawdown",
    "quant_score",
    "role_score",
    "selection_score",
]

selection_display_formatted[percentage_cols] = (
    selection_display_formatted[percentage_cols] * 100
)

selection_display_formatted

In [ ]:
review_table = selection_ranked[
    selection_ranked["decision"] == "Review"
][
    [
        "name",
        "role",
        "role_score",
        "actual_weight",
        "quant_score",
        "selection_score",
        "decision"
    ]
].copy()

review_table

In [ ]:
candidate_table = selection_ranked[
    selection_ranked["decision"] == "Candidate"
][
    [
        "name",
        "role",
        "role_score",
        "quant_score",
        "selection_score",
        "decision"
    ]
].copy()

candidate_table

## 12. Visualisation

In [ ]:
plot_df = selection_ranked.dropna(subset=["selection_score"]).copy()

plt.figure(figsize=(12, 5))

plt.bar(
    plot_df["name"],
    plot_df["selection_score"] * 100
)

plt.title("Asset Selection Score")
plt.xlabel("Asset")
plt.ylabel("Selection Score (%)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 13. Decision Summary

In [ ]:
decision_summary = (
    selection_ranked
    .groupby("decision")
    .size()
    .reset_index(name="n_assets")
    .sort_values("n_assets", ascending=False)
)

decision_summary

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    decision_summary["decision"],
    decision_summary["n_assets"]
)

plt.title("Asset Selection Decisions")
plt.xlabel("Decision")
plt.ylabel("Number of Assets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 14. Exports

In [ ]:
selection_ranked.to_csv(
    DATA_PATH / "asset_selection_ranking.csv",
    index=False
)

selection_display_formatted.to_csv(
    DATA_PATH / "asset_selection_ranking_display.csv",
    index=False
)

decision_summary.to_csv(
    DATA_PATH / "asset_selection_decision_summary.csv",
    index=False
)

review_table.to_csv(
    DATA_PATH / "asset_selection_review_table.csv",
    index=False
)

candidate_table.to_csv(
    DATA_PATH / "asset_selection_candidate_table.csv",
    index=False
)

print("Asset selection exports saved.")